<a href="https://colab.research.google.com/github/Divyesh-SP/DSA0302-NATURAL-LANGUAGE-PROCESSING/blob/main/NLP_CO1_Coding_Ch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
#Tokenization using a Regular Expression
import re

n = 4

pattern = r"[A-Za-z]+(?:['-][A-Za-z]+)*"

for _ in range(n):
    line = input()
    tokens = re.findall(pattern, line)

    if tokens:
        print(len(tokens), *tokens)
    else:
        print(0)


4
0
The dog's tail wagged -- twice!
5 The dog's tail wagged twice
state-of-the-art NLP systems, circa 2026.
4 state-of-the-art NLP systems circa
12345 ... ???
0


In [14]:
#Simulating a Finite State Automaton
n, k = map(int, input().split())
alphabet = input().split()
start = int(input())

data = list(map(int, input().split()))
m = data[0]
accepting = set(data[1:1 + m])

t = int(input())
transitions = {}

for _ in range(t):
    p, c, r = input().split()
    p, r = int(p), int(r)

    key = (p, c)
    if key not in transitions:
        transitions[key] = set()

    transitions[key].add(r)

q = int(input())

for _ in range(q):
    string = input()
    if string == "#":
        string = ""

    current_states = {start}

    for char in string:
        next_states = set()

        for state in current_states:
            next_states.update(
                transitions.get((state, char), set())
            )

        current_states = next_states

        if not current_states:
            break

    if current_states & accepting:
        print("ACCEPT")
    else:
        print("REJECT")


5 3
b a !
0
1 4
5
0 b 1
1 a 2
2 a 3
3 a 3
3 ! 4
5
baa!
ACCEPT
baaa!
ACCEPT
ba!
REJECT
b
REJECT
baa
REJECT


In [15]:
#Inflectional Morphology Generator
def is_vowel(ch):
    return ch in "aeiou"


def is_doublable(word):
    if len(word) < 3:
        return False

    a, b, c = word[-3], word[-2], word[-1]

    if is_vowel(a) or not is_vowel(b) or is_vowel(c):
        return False

    if c in "wxy":
        return False

    vowel_groups = 0
    in_vowel_group = False

    for ch in word:
        if is_vowel(ch):
            if not in_vowel_group:
                vowel_groups += 1
            in_vowel_group = True
        else:
            in_vowel_group = False

    return vowel_groups == 1


def inflect(word, feature):
    if feature in ("N+PL", "V+3SG"):
        if word.endswith(("s", "x", "z", "ch", "sh")):
            return word + "es"

        if (len(word) >= 2 and word.endswith("o")
                and not is_vowel(word[-2])):
            return word + "es"

        if (len(word) >= 2 and word.endswith("y")
                and not is_vowel(word[-2])):
            return word[:-1] + "ies"

        return word + "s"

    if feature == "V+PROG":
        if (len(word) >= 3 and word.endswith("e")
                and not is_vowel(word[-2])):
            return word[:-1] + "ing"

        if is_doublable(word):
            return word + word[-1] + "ing"

        return word + "ing"

    suffix = {
        "V+PAST": "ed",
        "ADJ+COMP": "er",
        "ADJ+SUP": "est"
    }[feature]

    if word.endswith("e"):
        return word + suffix[1:]

    if (len(word) >= 2 and word.endswith("y")
            and not is_vowel(word[-2])):
        return word[:-1] + "i" + suffix

    if is_doublable(word):
        return word + word[-1] + suffix

    return word + suffix


L = int(input())
lexicon = {}

for _ in range(L):
    stem, feature, form = input().split()
    lexicon[(stem, feature)] = form

Q = int(input())

for _ in range(Q):
    stem, feature = input().split()

    if (stem, feature) in lexicon:
        print(lexicon[(stem, feature)])
    else:
        print(inflect(stem, feature))


3
goose N+PL geese
go V+PAST went
good ADJ+COMP better
8
cat N+PL
cats
church N+PL
churches
baby N+PL
babies
stop V+PROG
stopping
make V+PROG
making
carry V+PAST
carried
goose N+PL
geese
go V+PAST
went


In [16]:
#Finite-State Morphological Parsing
def is_vowel(ch):
    return ch in "aeiou"


def is_doublable(word):
    if len(word) < 3:
        return False

    a, b, c = word[-3], word[-2], word[-1]

    if is_vowel(a) or not is_vowel(b) or is_vowel(c):
        return False

    if c in "wxy":
        return False

    groups = 0
    inside = False

    for ch in word:
        if is_vowel(ch):
            if not inside:
                groups += 1
            inside = True
        else:
            inside = False

    return groups == 1


def inflect(word, feature):
    if feature in ("PL", "3SG"):
        if word.endswith(("s", "x", "z", "ch", "sh")):
            return word + "es"

        if (len(word) >= 2 and word.endswith("o")
                and not is_vowel(word[-2])):
            return word + "es"

        if (len(word) >= 2 and word.endswith("y")
                and not is_vowel(word[-2])):
            return word[:-1] + "ies"

        return word + "s"

    if feature == "PROG":
        if (len(word) >= 3 and word.endswith("e")
                and not is_vowel(word[-2])):
            return word[:-1] + "ing"

        if is_doublable(word):
            return word + word[-1] + "ing"

        return word + "ing"

    suffix = {
        "PAST": "ed",
        "COMP": "er",
        "SUP": "est"
    }[feature]

    if word.endswith("e"):
        return word + suffix[1:]

    if (len(word) >= 2 and word.endswith("y")
            and not is_vowel(word[-2])):
        return word[:-1] + "i" + suffix

    if is_doublable(word):
        return word + word[-1] + suffix

    return word + suffix


S = int(input())
lexicon = []

for _ in range(S):
    stem, pos = input().split()
    lexicon.append((stem, pos))

Q = int(input())

features = {
    "N": ["PL"],
    "V": ["3SG", "PROG", "PAST"],
    "ADJ": ["COMP", "SUP"]
}

analyses_by_form = {}

for stem, pos in lexicon:
    # A bare stem is also a valid analysis.
    analyses_by_form.setdefault(stem, set()).add(
        stem + "+" + pos
    )

    for feature in features[pos]:
        form = inflect(stem, feature)
        analysis = stem + "+" + pos + "+" + feature

        analyses_by_form.setdefault(form, set()).add(analysis)

for _ in range(Q):
    surface = input()

    results = sorted(analyses_by_form.get(surface, set()))

    if results:
        print(", ".join(results))
    else:
        print("UNKNOWN")


5
cat N
watch N
watch V
study V
big ADJ
8
cats
cat+N+PL
watches
watch+N+PL, watch+V+3SG
watching
watch+V+PROG
studied
study+V+PAST
bigger
big+ADJ+COMP
cat
cat+N
biggest
big+ADJ+SUP
elephants
UNKNOWN


In [17]:
#The Porter Stemmer — Step 1
def is_consonant(word, i):
    ch = word[i]

    if ch in "aeiou":
        return False

    if ch == "y":
        if i == 0:
            return True
        return not is_consonant(word, i - 1)

    return True


def measure(word):
    # Count vowel-to-consonant transitions.
    m = 0

    for i in range(1, len(word)):
        if (not is_consonant(word, i - 1)
                and is_consonant(word, i)):
            m += 1

    return m


def has_vowel(word):
    for i in range(len(word)):
        if not is_consonant(word, i):
            return True
    return False


def ends_double_consonant(word):
    return (
        len(word) >= 2
        and word[-1] == word[-2]
        and is_consonant(word, len(word) - 1)
    )


def ends_cvc(word):
    if len(word) < 3:
        return False

    a, b, c = len(word) - 3, len(word) - 2, len(word) - 1

    return (
        is_consonant(word, a)
        and not is_consonant(word, b)
        and is_consonant(word, c)
        and word[-1] not in "wxy"
    )


def porter_step1(word):
    # Step 1a
    if word.endswith("sses"):
        word = word[:-2]
    elif word.endswith("ies"):
        word = word[:-2]
    elif word.endswith("ss"):
        pass
    elif word.endswith("s"):
        word = word[:-1]

    # Step 1b
    if word.endswith("eed"):
        stem = word[:-3]

        if measure(stem) > 0:
            word = stem + "ee"

    elif word.endswith("ed") and has_vowel(word[:-2]):
        word = word[:-2]

        # Cleanup after removing ed
        if word.endswith(("at", "bl", "iz")):
            word += "e"
        elif (ends_double_consonant(word)
              and word[-1] not in "lsz"):
            word = word[:-1]
        elif measure(word) == 1 and ends_cvc(word):
            word += "e"

    elif word.endswith("ing") and has_vowel(word[:-3]):
        word = word[:-3]

        # Cleanup after removing ing
        if word.endswith(("at", "bl", "iz")):
            word += "e"
        elif (ends_double_consonant(word)
              and word[-1] not in "lsz"):
            word = word[:-1]
        elif measure(word) == 1 and ends_cvc(word):
            word += "e"

    # Step 1c
    if word.endswith("y") and has_vowel(word[:-1]):
        word = word[:-1] + "i"

    return word


Q = int(input())

for _ in range(Q):
    word = input().strip()
    print(porter_step1(word))


8
caresses
caress
ponies
poni
cats
cat
feed
feed
agreed
agree
motoring
motor
hopping
hop
happy
happi
